# 02. Limpieza y selección de variables - Fashion Transparency Index 2023

### Objetivo del notebook

Preparar la base de datos integrada para el análisis exploratorio y la construcción de perfiles de transparencia corporativa, identificando los indicadores comparables entre empresas y seleccionando las variables que serán utilizadas en las etapas posteriores del proyecto.

### Descripción general

A partir de la base integrada construida en la etapa anterior, se revisó la naturaleza de los indicadores disponibles para identificar cuáles pueden compararse de manera consistente entre empresas.

Posteriormente, los indicadores que cumplieron los criterios de comparabilidad fueron transformados a un formato adecuado para el análisis de agrupamiento. Finalmente, se revisaron los valores faltantes, la variabilidad de las variables y la posible existencia de indicadores redundantes antes de generar la base definitiva para el análisis.

In [27]:
# Importamos las librerias necesarias

import pandas as pd
from pathlib import Path

In [28]:
# Definimos la ruta donde se encuentra la base procesada

ruta_datos = Path("/Users/avrilsalazar/Documents/FashionTransparency2023/processed")

In [29]:
# Cargamos la base integrada construida en la etapa anterior

base = pd.read_csv(
    ruta_datos / "base_integrada.csv"
)

# Mostramos su tamaño

print(f"Número de registros: {len(base):,}")
print(f"Número de variables: {base.shape[1]}")

Número de registros: 32,500
Número de variables: 12


In [30]:
# Creamos una copia de la base para realizar el proceso de limpieza

base_limpia = base.copy()

La base integrada obtenida en la etapa anterior contiene 32,500 observaciones y 12 variables. A partir de este punto, todas las etapas de limpieza y selección de variables se realizarán sobre una copia de la base original para conservar una versión íntegra de los datos procesados y facilitar la reproducibilidad del análisis.

### Revisión de los tipos de indicadores

In [31]:
# Cargamos el diccionario de indicadores

diccionario_indicadores = pd.read_excel(
    ruta_datos / "diccionario_indicadores.xlsx"
)

In [32]:
# Revisamos la distribución de los tipos de respuesta

diccionario_indicadores["Tipo de respuesta"].value_counts()

Tipo de respuesta
Sí / No               110
Lista de elementos     10
Puntaje                 6
Porcentaje              3
Texto                   1
Name: count, dtype: int64

Los 130 indicadores del Fashion Transparency Index 2023 se clasifican en cinco tipos de respuesta: variables binarias (Sí / No), listas de elementos, puntajes, indicadores de porcentaje y respuestas en texto. Esta clasificación permite distinguir los indicadores que comparten una estructura de medición de aquellos que requieren tratamientos particulares antes de incorporarse al análisis.

In [33]:
# Seleccionamos un indicador representativo de cada tipo de respuesta
# para revisar posteriormente cómo se encuentran almacenados sus valores.

tipos = ["Sí / No", "Lista de elementos", "Puntaje", "Porcentaje", "Texto"]

for tipo in tipos:

    ejemplo = (
        diccionario_indicadores.loc[
            diccionario_indicadores["Tipo de respuesta"] == tipo,
            "Indicador"
        ]
        .iloc[0]
    )

    print(f"\n{tipo}")
    print(f"Indicador: {ejemplo}")


Sí / No
Indicador: 1.5 Verified Sustainability Report

Lista de elementos
Indicador: 1.1 Own Operations Policies

Puntaje
Indicador: 1. Policy & Commitments Score

Porcentaje
Indicador: 3.1 Tier One Factory Disclosure

Texto
Indicador: 2.1 Identifies Lead Responsibility for Human Rights & Environmental Issues


In [34]:
# Consultamos un valor real de la base para cada tipo de respuesta.

# Esta revisión permite observar ejemplos de la forma en que
# las respuestas se encuentran almacenadas en la base.

tipos = [
    "Sí / No",
    "Lista de elementos",
    "Puntaje",
    "Porcentaje",
    "Texto"
]

for tipo in tipos:

    indicador = (
        diccionario_indicadores.loc[
            diccionario_indicadores["Tipo de respuesta"] == tipo,
            "Indicador"
        ].iloc[0]
    )

    print(f"\n{tipo}")
    print(f"Indicador: {indicador}")

    ejemplo = (
        base_limpia.loc[
            base_limpia["Metric"] == f"Fashion Revolution+{indicador}",
            "Value"
        ].dropna().iloc[0]
    )

    print(f"Ejemplo de respuesta: {ejemplo}")


Sí / No
Indicador: 1.5 Verified Sustainability Report
Ejemplo de respuesta: Yes

Lista de elementos
Indicador: 1.1 Own Operations Policies
Ejemplo de respuesta: Diversity & Inclusion, Anti-bribery Corruption & Presentation of False Information

Puntaje
Indicador: 1. Policy & Commitments Score
Ejemplo de respuesta: 6.2879067599067575

Porcentaje
Indicador: 3.1 Tier One Factory Disclosure
Ejemplo de respuesta: Type of products or services, Sex-disaggregated breakdown of workers at each site, % or number of migrant or contract workers, Discloses aggregate business volume and percentage of suppliers (assessed from 2022), Publishes that this list or map of tier one factories has been updated within the past 6 months, Discloses 95% or higher of tier one factories are included in the list/map, Certifications the facility has (if any) (2021 only), Address, Approximate number of workers at each site, Name of Facility, Supplier list contributed to the Open Apparel Registry (assessed from 2022),

### Selección de indicadores para el análisis

La revisión realizada muestra que los indicadores del Fashion Transparency Index 2023 representan distintos tipos de información. Mientras que algunos registran únicamente si una empresa divulga o no determinada información, otros contienen listas de elementos, puntajes agregados o respuestas descriptivas, por lo que no todos pueden compararse directamente bajo un mismo criterio.

El objetivo de esta etapa es construir una matriz empresa–indicador con variables que compartan una misma estructura de medición. Incorporar simultáneamente listas de elementos, puntajes y respuestas textuales requeriría metodologías de transformación diferentes para cada tipo de dato, lo que introduciría criterios de comparación distintos dentro de una misma matriz y dificultaría la interpretación de los grupos obtenidos.

Por ello, los **110 indicadores binarios** se consideran inicialmente los más adecuados para construir la matriz empresa–indicador, ya que todos comparten una misma escala de medición y permiten comparar de manera consistente si una empresa divulga o no determinada información. Antes de realizar su transformación numérica, se verificará que sus valores correspondan efectivamente a respuestas **Yes/No**, se revisarán los valores faltantes y se evaluará la variabilidad de cada indicador.



In [35]:
# Seleccionamos los indicadores clasificados como respuestas binarias.

indicadores_binarios = diccionario_indicadores.loc[
    diccionario_indicadores["Tipo de respuesta"] == "Sí / No",
    "Indicador"
]

print(f"Número de indicadores binarios: {len(indicadores_binarios)}")

Número de indicadores binarios: 110


In [36]:
# Revisamos los valores distintos registrados en los indicadores binarios.

valores_binarios = (
    base_limpia.loc[
        base_limpia["Metric"].isin(
            "Fashion Revolution+" + indicadores_binarios
        ),
        "Value"
    ]
    .dropna()
    .unique()
)

sorted(valores_binarios)

['No', 'Yes']

La revisión confirma que los indicadores seleccionados almacenan únicamente respuestas Yes y No, por lo que pueden transformarse de manera consistente a una representación numérica para construir la matriz empresa–indicador.

### Codificación numérica de los indicadores binarios

In [37]:
# Creamos una copia de la base para aplicar la codificación
# únicamente a los indicadores binarios.

base_binaria = base_limpia.copy()

# Identificamos los registros correspondientes a indicadores binarios.

mascara_binarios = (
    base_binaria["Metric"].isin(
        "Fashion Revolution+" + indicadores_binarios
    )
)

# Transformamos las respuestas binarias a una representación numérica.
# Yes indica que la empresa divulga la información (1),
# mientras que No indica que no la divulga (0).

base_binaria.loc[mascara_binarios, "Value"] = (
    base_binaria.loc[mascara_binarios, "Value"]
    .map({
        "Yes": 1,
        "No": 0
    })
)

In [38]:
# Verificamos la distribución de los valores después de aplicar
# la codificación numérica a los indicadores binarios.

base_binaria.loc[
    mascara_binarios,
    "Value"
].value_counts()

Value
0    20586
1     6914
Name: count, dtype: int64

La codificación produjo 20,586 registros con valor **0** y 6,914 con valor **1**. La suma de ambas categorías corresponde a los 27,500 registros esperados para los 110 indicadores evaluados en las 250 empresas, por lo que no se identifican valores faltantes en este subconjunto.

### Construcción de la matriz empresa–indicador

In [39]:
# Construimos una matriz donde cada fila representa una empresa
# y cada columna corresponde a un indicador binario.
# Los valores de la matriz indican si la empresa divulga (1)
# o no divulga (0) la información asociada a cada indicador.

matriz_empresas = (
    base_binaria.loc[mascara_binarios]
    .pivot(
        index="Company",
        columns="Metric",
        values="Value"
    )
)

In [40]:
# Revisamos las dimensiones de la matriz construida.

print(f"Número de empresas: {matriz_empresas.shape[0]}")
print(f"Número de indicadores: {matriz_empresas.shape[1]}")

Número de empresas: 250
Número de indicadores: 110


In [41]:
# Visualizamos las primeras filas de la matriz para verificar
# que la estructura sea la esperada.

matriz_empresas.head()

Metric,Fashion Revolution+1.5 Verified Sustainability Report,Fashion Revolution+Accountable Board Member Identified,Fashion Revolution+Affected Stakeholder Engagement in Remediation,Fashion Revolution+Approach to Defining Sustainable Materials,Fashion Revolution+Approach to Involving Women in Human Rights Due Diligence,Fashion Revolution+Commitment to Degrowth,Fashion Revolution+Commitment to Eliminate Hazardous Chemicals,Fashion Revolution+Decarbonisation Commitment,Fashion Revolution+Decarbonisation Progress,Fashion Revolution+Describes Environmental Due Diligence Process,...,Fashion Revolution+Supplier Incentives to Improve Impacts,Fashion Revolution+Supply Chain Policies Align with International Standards,Fashion Revolution+Supply Chain Policies Are Contractual,Fashion Revolution+Supply Chain Policies in Local Language,Fashion Revolution+Sustainable Materials Strategy,Fashion Revolution+Targets to Reduce Textiles Derived from Virgin Fossil Fuels,Fashion Revolution+Targets to Reduce Virgin Plastics,Fashion Revolution+Worker Representation on Board,Fashion Revolution+Zero Deforestation Commitment,Fashion Revolution+Zero Deforestation Progress
Company,,,,,,,,,,,,,,,,,,,,,
AJIO,0,1,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
ALDO,0,1,1,0,0,0,0,1,0,1,...,0,1,1,1,0,0,0,0,0,0
Abercrombie & Fitch,0,1,0,0,0,0,0,0,0,0,...,1,1,1,0,1,1,0,0,0,0
Adidas AG,1,1,1,1,0,0,1,1,1,1,...,1,1,1,1,1,1,0,1,0,0
Aeropostale Inc.,0,0,0,0,0,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0


En esta etapa únicamente se incorporan los indicadores binarios; los demás tipos de información se conservarán para su análisis e interpretación en etapas posteriores del estudio.

### Revisión de la variabilidad de los indicadores

In [42]:
# Calculamos la proporción de empresas que divulgan cada indicador.
# Como los indicadores están codificados en 0 y 1, la media representa
# el porcentaje de empresas que divulga cada indicador.

proporcion_divulgacion = matriz_empresas.mean()

# Revisamos los indicadores con menor proporción de divulgación.

proporcion_divulgacion.sort_values().head(10)

Metric
Fashion Revolution+Discloses Absolute Energy Reduction                                                           0.0
Fashion Revolution+Discloses Policy on Up-Front Supplier Payments                                              0.008
Fashion Revolution+Commitment to Degrowth                                                                      0.008
Fashion Revolution+Publishes Standard Supplier Agreement Template                                              0.008
Fashion Revolution+Publishes Percentage or Number of Workers Earning a Living Wage                             0.012
Fashion Revolution+Discloses Number of Collective Bargaining Agreements Providing Wages Above Legal Minimum    0.012
Fashion Revolution+Discloses Percentage of Workers Paid By Piece Rate                                          0.016
Fashion Revolution+Reports on Minimum Wage Paid for Daily / Piece Rate Workers                                 0.016
Fashion Revolution+Discloses Progress toward the payment 

In [43]:
# Revisamos los indicadores con mayor proporción de divulgación.

proporcion_divulgacion.sort_values(ascending=False).head(10)

Metric
Fashion Revolution+Supply Chain Policies Align with International Standards    0.752
Fashion Revolution+Plan for Improving Environmental Impacts                    0.716
Fashion Revolution+Supply Chain Policies Are Contractual                       0.712
Fashion Revolution+Describes Human Rights Due Diligence Process                0.676
Fashion Revolution+Grievance Mechanism - Direct Employees                      0.672
Fashion Revolution+Implementation of Board Level Accountability Described       0.66
Fashion Revolution+New Production Facility Criteria                            0.636
Fashion Revolution+Grievance Mechanism - Supply Chain Workers                  0.612
Fashion Revolution+Reports on Efforts to Improve Environmental Impacts         0.612
Fashion Revolution+Discloses Content of Scope 1, 2 and 3 Emissions             0.596
dtype: object

### Identificación de indicadores con baja variabilidad

In [44]:
# Identificamos los indicadores cuya proporción de divulgación
# es igual a 0 o igual a 1, ya que no presentan variabilidad
# entre las empresas.

indicadores_constantes = proporcion_divulgacion[
    (proporcion_divulgacion == 0) |
    (proporcion_divulgacion == 1)
]

print(f"Número de indicadores constantes: {len(indicadores_constantes)}")

indicadores_constantes

Número de indicadores constantes: 1


Metric
Fashion Revolution+Discloses Absolute Energy Reduction    0.0
dtype: object

Se identificó un único indicador sin variabilidad, ya que ninguna empresa reportó información para este criterio. Debido a que este indicador no contribuye a diferenciar empresas, será excluido de la matriz utilizada para el análisis de agrupamiento.

In [45]:
# Eliminamos el indicador sin variabilidad de la matriz que será
# utilizada en el análisis de agrupamiento.

matriz_empresas = matriz_empresas.drop(
    columns=indicadores_constantes.index
)

print(f"Número de indicadores: {matriz_empresas.shape[1]}")

Número de indicadores: 109


### Revisión de indicadores redundantes

In [46]:
# Identificamos indicadores con exactamente el mismo patrón
# de respuestas entre todas las empresas.

indicadores_redundantes = (
    matriz_empresas.T
    .duplicated()
)

print(
    f"Número de indicadores redundantes: "
    f"{indicadores_redundantes.sum()}"
)

Número de indicadores redundantes: 1


Se identificó un par de indicadores con un patrón de respuestas exactamente igual para las 250 empresas analizadas. Desde una perspectiva estadística, ambos contienen la misma información observada en la edición 2023 y su inclusión conjunta puede otorgar un mayor peso a este patrón durante el análisis de agrupamiento. No obstante, antes de decidir su tratamiento es necesario identificar cuáles son estos indicadores y revisar si corresponden a una misma métrica o a conceptos diferentes dentro del Fashion Transparency Index.

In [47]:
# Mostramos el indicador identificado como redundante.

matriz_empresas.columns[indicadores_redundantes]

Index(['Fashion Revolution+Reports on Minimum Wage Paid for Daily / Piece Rate Workers'], dtype='object', name='Metric')

In [48]:
# Identificamos con qué indicador coincide
# el indicador redundante.

indicador_redundante = matriz_empresas.columns[indicadores_redundantes][0]

duplicado = (
    matriz_empresas.T.loc[
        matriz_empresas.T.eq(matriz_empresas[indicador_redundante], axis=1).all(axis=1)
    ]
)

duplicado.index

Index(['Fashion Revolution+Discloses Percentage of Workers Paid By Piece Rate', 'Fashion Revolution+Reports on Minimum Wage Paid for Daily / Piece Rate Workers'], dtype='object', name='Metric')

El indicador "Reports on Minimum Wage Paid for Daily / Piece Rate Workers" (reporta el salario mínimo pagado a trabajadores remunerados por día o por pieza) presentó exactamente el mismo patrón de respuestas que "Discloses Percentage of Workers Paid By Piece Rate" (divulga el porcentaje de trabajadores remunerados mediante pago por pieza) en las 250 empresas analizadas. Aunque ambos indicadores contienen la misma información estadística en esta edición del índice, representan métricas conceptualmente diferentes. Por esta razón, se conservarán ambos en la matriz empresa–indicador para preservar la estructura original del Fashion Transparency Index. Esta redundancia deberá considerarse posteriormente durante la interpretación de los resultados del análisis de agrupamiento.

Al finalizar la etapa de limpieza se generan dos conjuntos de datos. El primero corresponde a la base integrada limpia, que conserva los indicadores binarios y no binarios, con excepción del indicador sin variabilidad identificado durante esta etapa, y será utilizada en el análisis exploratorio de datos. El segundo corresponde a la matriz empresa–indicador, integrada únicamente por los indicadores binarios seleccionados para el análisis de agrupamiento. De esta manera, cada etapa del estudio utiliza la estructura de datos más adecuada para sus objetivos.

### Incorporación de la categoría temática

Antes de exportar la base de datos limpia, se incorpora la categoría temática correspondiente a cada indicador utilizando el diccionario de indicadores elaborado durante la construcción de la base integrada. Esta información permitirá realizar comparaciones entre las categorías Environment, Social, Governance y Traceability durante el análisis exploratorio de datos.

In [49]:
# Cargamos el diccionario de indicadores elaborado durante
# la construcción de la base integrada
diccionario_indicadores = pd.read_excel(
    ruta_datos / "diccionario_indicadores.xlsx"
)

# Eliminamos la columna de categoría existente, que contenía
# la clasificación utilizada durante la organización inicial
# de las descargas (Environment, Social, Governance y Other),
# y la reemplazamos por la categoría temática definida
# en el diccionario de indicadores.
base_binaria = base_binaria.drop(
    columns="Categoria",
    errors="ignore"
)

# Incorporamos la categoría temática correspondiente a cada
# indicador utilizando el nombre del indicador como llave de unión.
# En la base la variable se llama "Metric" y en el diccionario
# "Indicador original".
base_binaria = base_binaria.merge(
    diccionario_indicadores[
        ["Indicador original", "Categoría temática"]
    ],
    left_on="Metric",
    right_on="Indicador original",
    how="left"
)

# Eliminamos la columna auxiliar utilizada para realizar la unión,
# ya que contiene la misma información que la variable "Metric"
base_binaria = base_binaria.drop(
    columns="Indicador original"
)

# Renombramos la categoría temática para mantener un nombre
# corto y consistente en el resto del proyecto
base_binaria = base_binaria.rename(
    columns={"Categoría temática": "Categoria"}
)

# Verificamos que cada indicador cuente con su categoría temática
base_binaria[["Metric", "Categoria"]].head()

,Metric,Categoria
0,Fashion Revolution+Fashion Transparency Index ...,Environment
1,Fashion Revolution+5. Spotlight Issues Score (...,Environment
2,Fashion Revolution+Zero Deforestation Progress,Environment
3,Fashion Revolution+Fashion Transparency Index ...,Environment
4,Fashion Revolution+5. Spotlight Issues Score (...,Environment


In [50]:
# Verificamos la distribución de las categorías temáticas
base_binaria["Categoria"].value_counts(dropna=False)

Categoria
Social          13250
Environment     11750
Governance       6500
Traceability     1000
Name: count, dtype: int64

### Exportación de la base de datos depurada y la matriz empresa-indicador

In [51]:
# Eliminamos de la base integrada el indicador sin variabilidad
# identificado durante el proceso de limpieza.

base_integrada_limpia = base_binaria[
    ~base_binaria["Metric"].isin([
        "Fashion Revolution+Discloses Absolute Energy Reduction"
    ])
]

# Exportamos la base integrada limpia para el análisis exploratorio.

base_integrada_limpia.to_csv(
    ruta_datos / "base_integrada_limpia.csv",
    index=False
)

In [52]:
# Exportamos la matriz empresa-indicador que será utilizada
# en el análisis de agrupamiento.

matriz_empresas.to_csv(
    ruta_datos / "matriz_empresas_clustering.csv"
)